# 3.1 The two-level map equation

In this notebook, we explore the map equation as an objective function and use it to compute the codelength of different partitions.

We also use Infomap to detect a network's community structure and compute the corresponding codelength

Because this notebook is intended as a learning resource, the code presented here is **not** optimised for speed.

In [ ]:
%matplotlib inline

import infomap
import json
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from _support import get_map_equation_example_network
from scipy.stats import entropy

In [ ]:
# use pastel colours later
sns.set_palette(sns.color_palette("colorblind"))

In [ ]:
# we use the "standard" map equation example network and plot it below
# the networks is weighted and undirected
G, pos = get_map_equation_example_network()

fig, ax = plt.subplots(1, 1, figsize = (4, 4))

base_color = np.array(sns.color_palette()[0][:3])
bright = tuple(0.5 * base_color + 0.5)

nx.draw_networkx_nodes(G = G, pos = pos, ax = ax,
                       node_color = G.number_of_nodes() * [sns.color_palette()[0]],
                       edgecolors = 'white', linewidths = 1.5)
nx.draw_networkx_edges(G = G, pos = pos, ax = ax, edgelist = sorted(G.edges),
                       width = [G.get_edge_data(*e)["weight"] for e in sorted(G.edges)],
                       arrows = True, connectionstyle = "arc3,rad=0.1",
                       edge_color = [bright] * G.number_of_edges())
nx.draw_networkx_labels(G = G, pos = pos, ax = ax, font_color = bright)

ax.axis("off")
plt.show()

In undirected networks, we can calculate the node visit rate for node $u$ as $p_u = \frac{s_u}{\sum_v s_v}$, where $s_u$ is node $u$'s strength, that is, the total weight of its links.

In [ ]:
# first, we save all nodes' strength in a dictionary
node_strengths = {u:G.degree(u, weight = "weight") for u in G.nodes()}
node_strengths

In [ ]:
# then, we calculate the nodes' visit rates
p = {u:s_u/sum(node_strengths.values()) for u,s_u in node_strengths.items()}
p

In a one-level partition $\mathsf{M}_1$ where all nodes are in the same module, the codelength is the entropy over the nodes' visit rates, $L\left(\mathsf{M}_1\right) = H\left(P\right) = - \sum_u p_u \log_2 p_u$.

In [ ]:
# we can put all nodes into the same module
M_1 = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]

# then, the codelength is simply the entropy over the nodes' visit rates
codelength = -sum([p_u * np.log2(p_u) for p_u in p.values()])
print(f"L(M_1) = {codelength:.4f} bits")

But if we look at the network, we see that there are four modules. We assign the nodes accordingly.

In [ ]:
M = [ [0, 1, 2, 3, 4, 5]
    , [6, 7, 8, 9, 10, 11, 12]
    , [13, 14, 15, 16]
    , [17, 18, 19, 20, 21, 22, 23, 24]
    ]

In [ ]:
# let's colour the network accordingly
fig, ax = plt.subplots(1, 1, figsize = (4, 4))

def brighten(color):
    c = np.array(list(color)[:3])
    return tuple(float(x) for x in 0.5 * c + 0.5)

nodelist     = []
node_colours = []
node_color_map = {}
for m, colour in zip(M, sns.color_palette()):
    for u in m:
        nodelist.append(u)
        node_colours.append(colour)
        node_color_map[u] = colour

# edge color: blend the two endpoint module colors, then brighten halfway to white
edge_colors = []
for u, v in sorted(G.edges):
    blended = (np.array(list(node_color_map[u])[:3]) + np.array(list(node_color_map[v])[:3])) / 2
    edge_colors.append(brighten(blended))

nx.draw_networkx_nodes(G = G, pos = pos, ax = ax, nodelist = nodelist, node_color = node_colours,
                       edgecolors = 'white', linewidths = 1.5)
nx.draw_networkx_edges(G = G, pos = pos, ax = ax, edgelist = sorted(G.edges),
                       width = [G.get_edge_data(*e)["weight"] for e in sorted(G.edges)],
                       arrows = True, connectionstyle = "arc3,rad=0.1", edge_color = edge_colors)

# draw labels per module with that module's brightened color
for m, colour in zip(M, sns.color_palette()):
    nx.draw_networkx_labels(G = G, pos = pos, ax = ax, labels = {u: u for u in m},
                            font_color = brighten(colour))

ax.axis("off")
plt.show()

For a two-level partition, we calculate the codelength with the two-level map equation, $L\left(\mathsf{M}\right) = q H\left(Q\right) + \sum_{\mathsf{m} \in \mathsf{M}} p_\mathsf{m} H\left(P_\mathsf{m}\right)$.

To calculate the two-level map equation, we need to know the module entry rates, $p_{\mathsf{m}\curvearrowleft} = \sum_{u \not \in \mathsf{m}, v \in \mathsf{m}} p_u p_{uv}$ with $p_{uv} = \frac{w_{uv}}{\sum_v w_{uv}}$.

In [ ]:
# first, we compute the transition rates because we will use them
transition_rates = dict()

for u in G.nodes:
    for v in G.neighbors(u):
        transition_rates[(u,v)] = G.get_edge_data(u, v)["weight"] / node_strengths[u]

transition_rates

In [ ]:
# to calculate the two-level map equation, we need to know the module entry rates
entry_rates = []

# we get them by looking up, for each module, how many links 
for m in M:
    m_entry = 0
    for v in m:
        for u in G.neighbors(v):
            if u not in m:
                m_entry += p[u] * transition_rates[u,v]
    entry_rates.append(m_entry)

entry_rates

In [ ]:
# then, we're ready to calculate the codelength
codelength = 0

# first, the codelength for the index-level codebook.
Q = entry_rates
q = sum(Q)
codelength += q * entropy(Q, base = 2)

# then, we add the codelength for each module
for m, m_exit in zip(M, entry_rates):
    P_m = [p[u] for u in m] + [m_exit]
    p_m = sum(P_m)
    codelength += p_m * entropy(P_m, base = 2)

print(f"L(M) = {codelength:.4f} bits")

We confirm our result with Infomap.

In [ ]:
# we can use Infomap to compute the codelength for the one-level partition by using the no_infomap flag.

# create an Infomap instance
im = infomap.Infomap(silent = True, no_infomap = True)

# add the network
im.add_networkx_graph(G)

# run it
im.run()

print(f"L(M_1) = {im.codelength:.4f} bits")

In [ ]:
# or we can use Infomap to detect communities

# create an Infomap instance
im = infomap.Infomap(silent = True)

# add the network and get Infomap's internal names for the nodes
im.add_networkx_graph(G)

# run it
im.run()

# collect Infomap's modules
modules = dict()
for u,m in dict(im.modules).items():
    if m not in modules:
        modules[m] = set()
    modules[m].add(u)

print(f"Infomap detected modules: {[sorted(m) for m in modules.values()]}")
print(f"L(M) = {im.codelength:.4f} bits")

In [ ]:
# Maintenance smoke assertions
G, _ = get_map_equation_example_network()
assert G.number_of_nodes() == 25
assert G.number_of_edges() == 43
assert infomap.Infomap(silent=True) is not None
